# 03. Feature Normalization (Fixed)

**Goal**: Normalize gameplay features per player using Min-Max scaling to capture relative behavior (0-1 scale).

**Inputs**:
- `data/processed/2_cleaned_telemetry_for_modelling.csv`

**Outputs**:
- `data/processed/3_normalized_telemetry.csv`

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import os

INPUT_PATH = '../../data/processed/2_cleaned_telemetry_for_modelling.csv'
OUTPUT_PATH = '../../data/processed/3_normalized_telemetry.csv'

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Cleaned Data

In [2]:
print("Loading cleaned telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows. Unique Users: {df['userId'].nunique()}")
else:
    raise FileNotFoundError(f"File not found at {INPUT_PATH}")

Loading cleaned telemetry data...
Loaded 2838 rows. Unique Users: 40


## 3. Identify Numeric Features

In [3]:
# Columns to Exclude
EXCLUDE = ['_id', 'userId', 'timestamp', 'session', 'death_count', 'time_since_start']

# Identify Numeric Columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Filter to get Feature Cols
feature_cols = [c for c in numeric_cols if c not in EXCLUDE and not c.startswith('rawJson')]

print(f"Features to Normalize ({len(feature_cols)}):")
print(feature_cols)

Features to Normalize (12):
['itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'enemiesHit', 'damageDone', 'timeInCombat', 'distanceTraveled', 'timeOutOfCombat', 'timeSprinting', 'kills', '__v_x', '__v_y']


## 4. Per-Player Min-Max Scaling
We use standard `groupby().apply()` without `tqdm` to ensure compatibility and stability.

In [4]:
def normalize_group(group):
    """Applies Min-Max scaling to a user's dataframe group."""
    for col in feature_cols:
        if col in group.columns:
            min_v = group[col].min()
            max_v = group[col].max()
            if max_v > min_v:
                group[col] = (group[col] - min_v) / (max_v - min_v)
            else:
                group[col] = 0.0
    return group

print("Normalizing per user...")
df_norm = df.groupby('userId', group_keys=False).apply(normalize_group)

# Ensure userId present
if 'userId' not in df_norm.columns:
    df_norm['userId'] = df['userId']

print("Normalization complete.")

Normalizing per user...
Normalization complete.


## 5. Verification & Export

In [5]:
# Safe verification and save
if 'userId' in df_norm.columns and not df_norm.empty:
    sample_user = df_norm['userId'].iloc[0]
    print(f"--- Verification (User {sample_user}) ---")
    print("Feature Ranges (Should be Min 0.0, Max 1.0 or 0.0):")
    print(df_norm[df_norm['userId'] == sample_user][feature_cols].agg(['min', 'max']))
else:
    print('No rows or userId missing; skipping range check.')

df_norm.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved normalized data to: {OUTPUT_PATH}")

--- Verification (User 6974892348d53c4152cf1421) ---
Feature Ranges (Should be Min 0.0, Max 1.0 or 0.0):
     itemsCollected  pickupAttempts  timeNearInteractables  enemiesHit  \
min             0.0             0.0                    0.0         0.0   
max             1.0             1.0                    1.0         1.0   

     damageDone  timeInCombat  distanceTraveled  timeOutOfCombat  \
min         0.0           0.0               0.0              0.0   
max         1.0           1.0               1.0              1.0   

     timeSprinting  kills  __v_x  __v_y  
min            0.0    0.0    0.0    0.0  
max            1.0    1.0    0.0    0.0  

Saved normalized data to: data/processed/3_normalized_telemetry.csv
